# DESC ELAsTiCC2 — SALT3 light-curve fitting with sncosmo

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-07
- **based on** : `02_sncosmo/02_elasticc2_fitsalt2_lightcurves.ipynb`

## Purpose

This notebook fits SNIa light curves from the ELAsTiCC2 training sample using the
**SALT3** model provided by `sncosmo`.  
The five SALT3 parameters — `z`, `t0`, `x0`, `x1`, `c` — are fitted **simultaneously
across all six LSST bands** (u, g, r, i, z, y) without any per-band renormalisation.

## SALT3 vs SALT2 — what changes?

SALT3 (Kenworthy et al. 2021, ApJ 923 265) has the **same parametric form** as SALT2:

$$
F(t,\lambda) = x_0\,\bigl[M_0(t,\lambda) + x_1\,M_1(t,\lambda)\bigr]\,
               10^{-0.4\,CL(\lambda)\,c}
$$

The free parameters are therefore **identical**: `z`, `t0`, `x0`, `x1`, `c`.  
There are **no additional parameters** compared to SALT2.

The key improvements of SALT3 over SALT2 are:

| Feature | SALT2 (`salt2-extended`) | SALT3 (`salt3`) |
|---|---|---|
| Wavelength coverage | 2 000 – 9 200 Å (optical only; extended to ~11 500 Å in `salt2-extended` via an extrapolation, not a full training) | **2 000 – 11 000 Å** natively trained — covers LSST *y*-band (peak ~9 700 Å) without extrapolation |
| Uncertainty model | Errors given in **magnitude** space | Errors given in **flux** space (more physically motivated) |
| Training sample | SALT2.4: ~430 SNe (Betoule et al. 2014, JLA) | SALT3.K21: ~1083 SNe — 2.5× larger, better cross-calibrated |
| Open-source training code | No | Yes (SALTshaker) |
| LSST *y*-band suitability | Requires `salt2-extended` extrapolation | **Native** coverage |

For LSST analysis, `salt3` is therefore the **recommended model** for low-to-medium
redshift SNIa: it covers the rest-frame y-band (and i-band at $z \sim 0.1$–0.4) without
relying on extrapolated template flux.

An even further NIR extension (**SALT3-NIR**, Pierel et al. 2022) covers up to 2 μm
and is registered as `'salt3-nir'` in sncosmo. It is designed for Roman Space Telescope
observations and is **not** required for LSST.

### sncosmo source name

```python
sncosmo.Model(source='salt3')            # default version (K21)
sncosmo.Model(source='salt3', version='K21')  # explicit version
```

### References
- Kenworthy et al. 2021 — SALT3: https://doi.org/10.3847/1538-4357/ac30d8
- Pierel et al. 2022 — SALT3-NIR: https://doi.org/10.3847/1538-4357/ac93f9
- sncosmo documentation: https://sncosmo.readthedocs.io/en/stable/index.html
- ELAsTiCC2 dataset: DESC TD public data
- Notebook `02_sncosmo/02_elasticc2_fitsalt2_lightcurves.ipynb` — SALT2 baseline


## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
from matplotlib import pyplot as plt

import sncosmo

# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

## 1 · Parameters

In [ ]:
# ── Data selection ─────────────────────────────────────────────────────────────
OBJ_CLASS      = 'SNIa-SALT3'   # SNANA class label
Z_MIN          = 0.1            # redshift lower bound
Z_MAX          = 0.5            # redshift upper bound
FILE_NUM       = 1              # PHOT file index (1–40; None = all)
MIN_DETECTIONS = 8              # minimum detected points per object
DETECTED_ONLY  = True           # use only detected points (PHOTFLAG & photflag_detect)
N_CURVES       = 20             # number of events to fit and display
RANDOM_SEED    = 42

# ── SALT3 model ────────────────────────────────────────────────────────────────
# The sncosmo source name is 'salt3' (version 'K21', Kenworthy et al. 2021).
# Wavelength range: 2000–11000 Å — covers LSST y-band natively.
# Parameters: z, t0, x0, x1, c  (same as SALT2; no extra parameters).
# Key difference from SALT2: uncertainties modelled in flux space, not magnitude space.
SALT3_SOURCE   = 'salt3'            # sncosmo source name
SALT3_VERSION  = 'K21'              # Kenworthy et al. 2021 training
ZP             = 31.4               # zero-point used in SNANA / ELAsTiCC2 (AB system)
ZPSYS          = 'ab'

# LSST band names as understood by sncosmo (prefix 'lsst' + letter)
BANDS          = ['u', 'g', 'r', 'i', 'z', 'y']   # lower-case, as in sncosmo
BAND_PREFIX    = 'lsst'

# ── Data path ──────────────────────────────────────────────────────────────────
DATA_DIR   = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX = "ELASTICC2_TRAIN_02_"

# ── Plot colours per band (consistent with other notebooks in this series) ─────
BAND_COLORS = {
    'u': '#cc0ccc',
    'g': '#00cc44',
    'r': '#cc0000',
    'i': '#ff4400',
    'z': '#886600',
    'y': '#442200'
}

# ── Display ────────────────────────────────────────────────────────────────────
NCOLS = 4

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"SALT3 source : {SALT3_SOURCE}  (version {SALT3_VERSION})")
print(f"ZP={ZP}  zpsys={ZPSYS}")
print(f"Bands : {BANDS}")
print(f"N_CURVES={N_CURVES}")

## 2 · SALT3 model inspection and band registration check

We load the model, print its parameters and wavelength range, and verify
that all six LSST bandpasses are registered in sncosmo.

In [ ]:
# Load SALT3 model — sncosmo downloads model files on first use and caches them.
# The default version is K21; specify explicitly for reproducibility.
#salt3_model = sncosmo.Model(source=sncosmo.get_source(SALT3_SOURCE, version=SALT3_VERSION))
salt3_model = sncosmo.Model(source=sncosmo.get_source(SALT3_SOURCE))

print("=" * 60)
print(f"Model       : {salt3_model.source.name}  v{salt3_model.source.version}")
print(f"Source type : {type(salt3_model.source).__name__}")
print(f"Parameters  : {salt3_model.param_names}")
print(f"Defaults    : {dict(zip(salt3_model.param_names, salt3_model.parameters))}")
print(f"Phase range : {salt3_model.source.minphase():.1f} to "
      f"{salt3_model.source.maxphase():.1f} days (rest frame)")
print(f"Wave range  : {salt3_model.source.minwave():.0f} to "
      f"{salt3_model.source.maxwave():.0f} Å (rest frame)")
print("=" * 60)
print()
print("Note: SALT3 has the same 5 parameters as SALT2 (z, t0, x0, x1, c).")
print("      The improvement is in error modelling and wavelength coverage,")
print("      not in the number of free parameters.")

# ── Verify LSST band registration ─────────────────────────────────────────────
print("\nChecking LSST band registration in sncosmo:")
for b in BANDS:
    bname = BAND_PREFIX + b
    try:
        bp = sncosmo.get_bandpass(bname)
        wave_cen = bp.wave_eff
        print(f"  {bname:10s}  λ_eff={wave_cen:.0f} Å  ✓")
    except Exception as e:
        print(f"  {bname:10s}  NOT FOUND: {e}")

### 2.1 · Visualise SALT3 template SED vs SALT2

A quick comparison of the spectral templates at peak phase to illustrate
the extended wavelength coverage of SALT3 (particularly the y-band region).

In [ ]:
# Compare SALT3 and SALT2-extended templates at peak phase (t = t0 = 0)
# using default parameters (x0=1, x1=0, c=0, z=0)
wave_plot = np.linspace(2000, 12000, 1000)  # Å

# Reference LSST band effective wavelengths (observer frame, z=0)
band_wave_eff = {b: sncosmo.get_bandpass(BAND_PREFIX + b).wave_eff for b in BANDS}

# SALT3 SED at phase=0
salt3_for_plot = sncosmo.Model(
    #source=sncosmo.get_source(SALT3_SOURCE, version=SALT3_VERSION)
    source=sncosmo.get_source(SALT3_SOURCE)
)
salt3_for_plot.set(z=0, t0=0, x0=1.0, x1=0.0, c=0.0)

# SALT2-extended SED at phase=0 for comparison
salt2_for_plot = sncosmo.Model(source='salt2-extended')
salt2_for_plot.set(z=0, t0=0, x0=1.0, x1=0.0, c=0.0)

# Evaluate spectra — only within valid wavelength range of each model
salt3_wmin = salt3_for_plot.source.minwave()
salt3_wmax = salt3_for_plot.source.maxwave()
salt2_wmin = salt2_for_plot.source.minwave()
salt2_wmax = salt2_for_plot.source.maxwave()

w3 = wave_plot[(wave_plot >= salt3_wmin) & (wave_plot <= salt3_wmax)]
w2 = wave_plot[(wave_plot >= salt2_wmin) & (wave_plot <= salt2_wmax)]

f3 = salt3_for_plot.flux(0, w3)   # at phase = 0
f2 = salt2_for_plot.flux(0, w2)

# Normalise both SEDs to their peak
f3_norm = f3 / np.nanmax(f3)
f2_norm = f2 / np.nanmax(f2)

fig_sed, ax_sed = plt.subplots(figsize=(10, 4), tight_layout=True)
ax_sed.plot(w3, f3_norm, 'C0-',  lw=2.0, label='SALT3 (K21)', zorder=3)
ax_sed.plot(w2, f2_norm, 'C1--', lw=1.5, label='SALT2-extended', zorder=2, alpha=0.8)

# Mark LSST band effective wavelengths
for b in BANDS:
    weff = band_wave_eff[b]
    ax_sed.axvline(weff, color=BAND_COLORS[b], lw=1.2, ls=':', alpha=0.9)
    ax_sed.text(weff, 1.02, b, ha='center', va='bottom', fontsize=9,
                color=BAND_COLORS[b], fontweight='bold')

# Shade the SALT3 extension region (beyond SALT2-extended ~9200 Å)
ax_sed.axvspan(salt2_wmax, salt3_wmax, alpha=0.08, color='C0',
               label=f'SALT3 extension ({salt2_wmax:.0f}–{salt3_wmax:.0f} Å)')

ax_sed.set_xlim(1800, 12200)
ax_sed.set_ylim(-0.05, 1.15)
ax_sed.set_xlabel('Wavelength [Å]', fontsize=11)
ax_sed.set_ylabel('Normalised flux (phase=0)', fontsize=11)
ax_sed.set_title(
    'SALT3.K21 vs SALT2-extended — SED at peak phase (x0=1, x1=0, c=0, z=0)\n'
    'Vertical dotted lines: LSST band effective wavelengths',
    fontsize=10
)
ax_sed.legend(fontsize=9, loc='upper right')
plt.show()

print(f"\nSALT3 wavelength range : {salt3_wmin:.0f} – {salt3_wmax:.0f} Å")
print(f"SALT2-extended range   : {salt2_wmin:.0f} – {salt2_wmax:.0f} Å")
print(f"LSST y-band λ_eff      : {band_wave_eff['y']:.0f} Å")
print(f"  → y-band covered natively by SALT3: "
      f"{'YES' if band_wave_eff['y'] < salt3_wmax else 'NO'}")

## 3 · Load ELAsTiCC2 data

Same loading pattern as the other notebooks in this series.

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading HEAD for {OBJ_CLASS}...")
head  = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading truth for {OBJ_CLASS}...")
truth = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")
print(f"Columns: {list(all_ltcvs.columns)}")

In [ ]:
# ── Filter: redshift range + minimum detections ───────────────────────────────
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)

truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
subset = truth_counts[
    (truth_counts['ZCMB'] >= Z_MIN) &
    (truth_counts['ZCMB'] <  Z_MAX) &
    (truth_counts['ndetect'] >= MIN_DETECTIONS)
].copy()

print(f"{len(subset)} objects pass selection (z∈[{Z_MIN},{Z_MAX}), ndet≥{MIN_DETECTIONS}).")

## 4 · Helper: build an `astropy.Table` for sncosmo

`sncosmo.fit_lc` expects an `astropy.Table` with columns
`time`, `band`, `flux`, `fluxerr`, `zp`, `zpsys`.

The ELAsTiCC2 data store band names as single letters (e.g. `'r'`, `'Y'`);  
we prepend `'lsst'` to match the sncosmo bandpass registry, and normalise to lower-case
(ELAsTiCC2 uses capital `'Y'`, sncosmo uses lower-case `'y'`).

In [ ]:
def make_sncosmo_table(ltcv_df: pd.DataFrame,
                       detected_only: bool = True,
                       photflag_detect: int = None,
                       zp: float = ZP,
                       zpsys: str = ZPSYS,
                       band_prefix: str = BAND_PREFIX) -> astropy.table.Table:
    """Convert an ELAsTiCC2 light-curve DataFrame to an astropy.Table
    suitable for sncosmo.fit_lc.

    Parameters
    ----------
    ltcv_df         : DataFrame for a single SNID (columns MJD, FLUXCAL,
                      FLUXCALERR, BAND, PHOTFLAG)
    detected_only   : if True, keep only rows where PHOTFLAG & photflag_detect != 0
    photflag_detect : bitmask value for detections (from esr.photflag_detect)
    zp              : photometric zero-point
    zpsys           : zero-point system ('ab')
    band_prefix     : prefix to prepend to the band letter (e.g. 'lsst')

    Returns
    -------
    astropy.Table with columns: time, band, flux, fluxerr, zp, zpsys
    """
    df = ltcv_df.copy()

    # Keep only detected observations if requested
    if detected_only and photflag_detect is not None:
        df = df[(df['PHOTFLAG'] & photflag_detect) != 0]

    # Remove rows with non-positive flux error (bad measurements)
    df = df[df['FLUXCALERR'] > 0].copy()

    # Normalise band names: strip whitespace, lower-case, prepend prefix
    # ELAsTiCC2 uses 'Y'; sncosmo expects 'lssty'
    df['BAND'] = df['BAND'].str.strip().str.lower()
    df['band_sncosmo'] = band_prefix + df['BAND']

    table = astropy.table.Table({
        'time'    : df['MJD'].values.astype(float),
        'band'    : df['band_sncosmo'].values,
        'flux'    : df['FLUXCAL'].values.astype(float),
        'fluxerr' : df['FLUXCALERR'].values.astype(float),
        'zp'      : np.full(len(df), zp, dtype=float),
        'zpsys'   : np.full(len(df), zpsys),
    })
    return table


print("make_sncosmo_table helper ready.")

## 5 · Helper: fit a single event with SALT3

We use `sncosmo.fit_lc` (native sncosmo fitter, Minuit/iminuit or scipy fallback).

**Free parameters** : `t0`, `x0`, `x1`, `c`  
**Fixed parameter** : `z` (set to truth `ZCMB`; optionally freed via `fit_z=True`)

The parameter `x0` is the overall amplitude (luminosity-like), `x1` is the SALT3
stretch (light-curve width), and `c` is the colour (SED tilt / dust).  
These are exactly the same free parameters as SALT2.

SALT3 improves the **uncertainties** on the fitted parameters because its internal
error model is in flux space: the covariance matrix returned by sncosmo will generally
be more accurate than what SALT2 provides.

In [ ]:
def fit_salt3_event(ltcv_df: pd.DataFrame,
                    z_true: float,
                    fit_z: bool = False,
                    photflag_detect: int = None) -> dict:
    """Fit a single ELAsTiCC2 SNIa event with SALT3 using sncosmo.

    SALT3 parameters are identical to SALT2: z, t0, x0, x1, c.
    The improvement over SALT2 lies in the error model (flux space)
    and the extended wavelength coverage (2000–11000 Å).

    Parameters
    ----------
    ltcv_df         : DataFrame for a single SNID
    z_true          : truth redshift (ZCMB) used to initialise the model
    fit_z           : if True, redshift is also a free parameter
    photflag_detect : bitmask for detections

    Returns
    -------
    dict with keys:
        'success', 'result' (sncosmo result object), 'fitted_model',
        'table' (astropy table), 'z', 't0', 'x0', 'x1', 'c',
        'chi2', 'ndof', 'chi2_red', 'vcov' (parameter covariance matrix),
        'param_errors' (1-sigma uncertainties from covariance), 'message'
    """
    # Build sncosmo observation table
    try:
        obs = make_sncosmo_table(
            ltcv_df,
            detected_only=DETECTED_ONLY,
            photflag_detect=photflag_detect
        )
    except Exception as e:
        return {'success': False, 'message': f'Table creation failed: {e}'}

    if len(obs) < 5:
        return {'success': False, 'message': 'Not enough data points after filtering'}

    # Initialise a fresh SALT3 model instance
    model = sncosmo.Model(
        #source=sncosmo.get_source(SALT3_SOURCE, version=SALT3_VERSION)
        source=sncosmo.get_source(SALT3_SOURCE)
    )

    # Initial parameter guess: t0 from brightest observation
    t0_guess = float(obs['time'][np.argmax(obs['flux'])])
    model.set(z=z_true, t0=t0_guess, x0=1e-4, x1=0.0, c=0.0)

    # Parameters to fit (z fixed by default)
    vparam_names = ['t0', 'x0', 'x1', 'c']
    bounds = {
        't0' : (t0_guess - 30.0, t0_guess + 30.0),
        'x0' : (1e-8, 1.0),
        'x1' : (-5.0, 5.0),
        'c'  : (-0.5, 0.5),
    }
    if fit_z:
        vparam_names = ['z', 't0', 'x0', 'x1', 'c']
        bounds['z'] = (max(z_true - 0.1, 0.001), z_true + 0.1)

    # Run fit with sncosmo.fit_lc
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result, fitted_model = sncosmo.fit_lc(
                obs, model,
                vparam_names=vparam_names,
                bounds=bounds,
                minsnr=0.0,    # include all detections regardless of S/N
                warn=False
            )
    except Exception as e:
        return {'success': False, 'message': f'Fit failed: {e}', 'table': obs}

    chi2     = result.chisq
    ndof     = result.ndof
    chi2_red = chi2 / max(ndof, 1)

    # Extract parameter uncertainties from covariance matrix
    vcov = result.covariance  # shape (n_vparam, n_vparam) or None
    param_errors = {}
    if vcov is not None:
        diag = np.diag(vcov)
        for i, pname in enumerate(vparam_names):
            param_errors[pname] = float(np.sqrt(max(diag[i], 0.0)))

    return {
        'success'       : True,
        'result'        : result,
        'fitted_model'  : fitted_model,
        'table'         : obs,
        'z'             : float(fitted_model['z']),
        't0'            : float(fitted_model['t0']),
        'x0'            : float(fitted_model['x0']),
        'x1'            : float(fitted_model['x1']),
        'c'             : float(fitted_model['c']),
        'chi2'          : float(chi2),
        'ndof'          : int(ndof),
        'chi2_red'      : float(chi2_red),
        'vcov'          : vcov,
        'param_errors'  : param_errors,
        'message'       : result.message,
    }


print("SALT3 single-event fitter ready.")

## 6 · Select events and run SALT3 fits

In [ ]:
# ── Sample N_CURVES events ─────────────────────────────────────────────────────
n_avail = min(N_CURVES, len(subset))
chosen_idx   = rng.choice(len(subset), size=n_avail, replace=False)
chosen_snids = subset['SNID'].values[chosen_idx]
print(f"Selected {n_avail} SNIDs for SALT3 fitting.")

In [ ]:
# ── Run SALT3 fits ─────────────────────────────────────────────────────────────
fit_results = {}

for snid in chosen_snids:
    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]

    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    res = fit_salt3_event(
        ltcv,
        z_true=z_true,
        fit_z=False,
        photflag_detect=esr.photflag_detect
    )
    fit_results[snid] = res

    if res['success']:
        # Show fitted values with 1-sigma errors when available
        pe = res.get('param_errors', {})
        x1_err = pe.get('x1', float('nan'))
        c_err  = pe.get('c',  float('nan'))
        print(
            f"  SNID {snid:8d}  ✓  "
            f"z={res['z']:.4f}  t0={res['t0']:.2f}  "
            f"x0={res['x0']:.3e}  "
            f"x1={res['x1']:+.3f}±{x1_err:.3f}  "
            f"c={res['c']:+.3f}±{c_err:.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}"
        )
    else:
        print(f"  SNID {snid:8d}  ✗  {res['message']}")

## 7 · Plot: multi-band light curves with SALT3 fit

For each event: raw flux data points (error bars, colour per band) +
SALT3 model curve (same colour, solid line).  
Abscissa: $t - t_0$ (days from peak).  
**No renormalisation**: absolute FLUXCAL scale.

In [ ]:
def plot_salt3_fit(ax, snid: int, res: dict, z_true: float):
    """Plot raw data + SALT3 model for one event.

    Parameters
    ----------
    ax      : matplotlib Axes
    snid    : SNANA object identifier
    res     : dict returned by fit_salt3_event
    z_true  : truth redshift for the subplot title
    """
    if not res.get('success'):
        ax.set_title(
            f"SNID {snid}\nFit failed\n{res.get('message', '')[:60]}",
            fontsize=8, color='red'
        )
        return

    obs          = res['table']          # astropy.Table
    fitted_model = res['fitted_model']   # sncosmo Model (SALT3)
    t0           = res['t0']
    chi2_red     = res['chi2_red']
    pe           = res.get('param_errors', {})

    # Dense time grid for smooth model curves
    t_min_data = float(obs['time'].min())
    t_max_data = float(obs['time'].max())
    t_dense    = np.linspace(t_min_data - 10, t_max_data + 10, 400)

    for b in BANDS:
        bname = BAND_PREFIX + b
        color = BAND_COLORS.get(b, 'gray')

        # ── Data points ────────────────────────────────────────────────────────
        mask = np.array(obs['band']) == bname
        if mask.sum() > 0:
            ax.errorbar(
                obs['time'][mask] - t0,
                obs['flux'][mask],
                yerr=obs['fluxerr'][mask],
                color=color, linestyle='None',
                marker='o', markersize=4, capsize=2,
                label=b, zorder=3
            )

        # ── SALT3 model curve ──────────────────────────────────────────────────
        try:
            f_model = fitted_model.bandflux(bname, t_dense, zp=ZP, zpsys=ZPSYS)
            valid = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(
                    t_dense[valid] - t0, f_model[valid],
                    color=color, lw=1.5, ls='-', zorder=2
                )
        except Exception:
            pass

    ax.axhline(0.0, color='k', lw=0.5, ls='--')

    x1_err = pe.get('x1', float('nan'))
    c_err  = pe.get('c',  float('nan'))
    ax.set_title(
        f"SNID {snid}  z={z_true:.3f}\n"
        f"x1={res['x1']:+.2f}±{x1_err:.2f}  "
        f"c={res['c']:+.2f}±{c_err:.2f}  "
        f"χ²/dof={chi2_red:.2f}",
        fontsize=8
    )
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=8)
    ax.set_ylabel("FLUXCAL", fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.legend(fontsize=6, ncol=3, loc='upper right')


# ── Draw grid of plots ─────────────────────────────────────────────────────────
nrows = math.ceil(n_avail / NCOLS)
fig, axes = plt.subplots(
    nrows, NCOLS,
    figsize=(5.5 * NCOLS, 4.2 * nrows),
    tight_layout=True
)
axes_flat = np.array(axes).flatten()

for idx, snid in enumerate(chosen_snids):
    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    plot_salt3_fit(axes_flat[idx], snid, fit_results[snid], z_true)

for idx in range(n_avail, len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    f"SALT3 (K21) fits — {OBJ_CLASS}  |  z ∈ [{Z_MIN},{Z_MAX})  |  "
    f"ndet ≥ {MIN_DETECTIONS}\n"
    f"Model: {SALT3_SOURCE} v{SALT3_VERSION}  |  free params: t0, x0, x1, c  "
    f"|  z fixed to truth",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()

## 8 · Summary table of fitted SALT3 parameters

Includes 1-sigma uncertainties from the covariance matrix returned by `sncosmo.fit_lc`.

In [ ]:
rows = []
for snid in chosen_snids:
    res   = fit_results[snid]
    z_row = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan
    pe    = res.get('param_errors', {})

    row = {
        'SNID'    : snid,
        'z_true'  : round(z_true, 4),
        'success' : res.get('success', False),
    }
    if res.get('success'):
        row.update({
            'z_fit'    : round(res['z'],       4),
            't0'       : round(res['t0'],       2),
            'x0'       : float(f"{res['x0']:.4e}"),
            'x1'       : round(res['x1'],       3),
            'x1_err'   : round(pe.get('x1', float('nan')), 3),
            'c'        : round(res['c'],        3),
            'c_err'    : round(pe.get('c',  float('nan')), 3),
            'chi2'     : round(res['chi2'],     2),
            'ndof'     : res['ndof'],
            'chi2_red' : round(res['chi2_red'], 3),
        })
    rows.append(row)

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

## 9 · Distribution of SALT3 parameters

Histograms of `x1`, `c`, χ²/dof, and the parameter uncertainties σ(x1), σ(c).

In [ ]:
df_ok = df_results[df_results['success']].copy()

params = ['x1', 'c', 'chi2_red', 'x1_err', 'c_err']
labels = {
    'x1'      : r'SALT3 $x_1$ (stretch)',
    'c'       : r'SALT3 $c$ (colour)',
    'chi2_red': r'$\chi^2$/dof',
    'x1_err'  : r'$\sigma(x_1)$ from fit',
    'c_err'   : r'$\sigma(c)$ from fit',
}
colors_hist = ['C0', 'C1', 'C2', 'C3', 'C4']

fig2, axes2 = plt.subplots(1, len(params),
                            figsize=(4.5 * len(params), 3.5),
                            tight_layout=True)

for ax, par, col in zip(axes2, params, colors_hist):
    vals = df_ok[par].dropna()
    if len(vals) == 0:
        ax.set_title(f'{par}\nno data', fontsize=9)
        continue
    ax.hist(vals, bins=10, color=col, edgecolor='white')
    ax.axvline(np.median(vals), color='k', ls='--', lw=1.5,
               label=f'median={np.median(vals):.3g}')
    ax.set_xlabel(labels[par], fontsize=10)
    ax.set_ylabel('N events', fontsize=10)
    ax.set_title(f'σ = {np.std(vals):.3g}', fontsize=9)
    ax.legend(fontsize=8)

fig2.suptitle(
    f"SALT3 parameter distributions  —  {OBJ_CLASS}  "
    f"(N={len(df_ok)} events,  model={SALT3_SOURCE} v{SALT3_VERSION})",
    fontsize=11
)
plt.show()

## 10 · Hubble diagram: distance modulus vs redshift

Tripp standardisation formula:

$$
\mu = m_B^* - M_B + \alpha\, x_1 - \beta\, c
$$

with $\alpha = 0.14$, $\beta = 3.1$ (Betoule et al. 2014 / Pantheon values).

The B-band apparent magnitude is derived from $x_0$:  
$m_B^* = -2.5\log_{10}(x_0) + \mathrm{const}$

The constant depends on the model normalisation.  For SALT3.K21 we use the same
value as for SALT2.4 (10.635) to first order — a precise determination would require
an absolute calibration step (e.g. Betoule 2014 or Brout et al. 2022).
The diagram is therefore shown **relative** (up to $M_B$).

The scatter around the reference ΛCDM curve reflects the intrinsic SNIa scatter
plus measurement noise; a reduced scatter compared to SALT2 is expected from
the improved error model.

In [ ]:
ALPHA_TRIPP = 0.14
BETA_TRIPP  = 3.10
MB_CONST    = 10.635   # SALT3.K21 normalisation (same as SALT2.4 to first order)

df_ok = df_ok.copy()
df_ok['mB_star']  = -2.5 * np.log10(df_ok['x0'].astype(float)) + MB_CONST
df_ok['mu_tripp'] = (df_ok['mB_star']
                     + ALPHA_TRIPP * df_ok['x1']
                     - BETA_TRIPP  * df_ok['c'])

fig3, ax3 = plt.subplots(figsize=(7, 4.5), tight_layout=True)
sc = ax3.scatter(
    df_ok['z_true'], df_ok['mu_tripp'],
    c=df_ok['chi2_red'], cmap='viridis_r',
    vmin=0, vmax=5,
    s=40, edgecolors='k', linewidths=0.3, zorder=3
)
plt.colorbar(sc, ax=ax3, label=r'$\chi^2$/dof')

# Flat ΛCDM reference
try:
    from astropy.cosmology import FlatLambdaCDM
    cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
    z_ref  = np.linspace(max(Z_MIN, 0.01), Z_MAX, 300)
    mu_ref = cosmo.distmod(z_ref).value
    ax3.plot(z_ref, mu_ref, 'r--', lw=1.5,
             label=r'Flat ΛCDM ($H_0$=70, $\Omega_m$=0.3)')
    ax3.legend(fontsize=9)
except Exception:
    pass

ax3.set_xlabel('Redshift z (truth)', fontsize=11)
ax3.set_ylabel(r'$\mu_{\rm Tripp}$ (relative)', fontsize=11)
ax3.set_title(
    f"Hubble diagram — {OBJ_CLASS}  ({len(df_ok)} events)\n"
    rf"Model: SALT3.K21  |  "
    rf"$\mu = m_B^* + {ALPHA_TRIPP}\,x_1 - {BETA_TRIPP}\,c$",
    fontsize=11
)
plt.show()

# ── Hubble residuals ───────────────────────────────────────────────────────────
try:
    df_ok['mu_lcdm'] = cosmo.distmod(df_ok['z_true'].values).value
    df_ok['mu_resid'] = df_ok['mu_tripp'] - df_ok['mu_lcdm']

    fig4, ax4 = plt.subplots(figsize=(7, 3), tight_layout=True)
    ax4.scatter(df_ok['z_true'], df_ok['mu_resid'],
                c=df_ok['chi2_red'], cmap='viridis_r',
                vmin=0, vmax=5, s=35, edgecolors='k', linewidths=0.3)
    ax4.axhline(0.0, color='r', ls='--', lw=1.5)
    rms = np.std(df_ok['mu_resid'].dropna())
    ax4.set_xlabel('Redshift z (truth)', fontsize=11)
    ax4.set_ylabel(r'$\Delta\mu = \mu_{\rm Tripp} - \mu_{\Lambda CDM}$', fontsize=10)
    ax4.set_title(
        f"Hubble residuals — SALT3.K21  |  RMS = {rms:.3f} mag",
        fontsize=11
    )
    plt.show()
    print(f"Hubble residual RMS = {rms:.4f} mag  (N={len(df_ok)} events)")
except Exception as e:
    print(f"Could not compute residuals: {e}")

## Summary

### SALT3 parameters fitted in this notebook

| Parameter | Status | Physical meaning |
|-----------|--------|------------------|
| `z`  | **fixed** to truth `ZCMB` | Cosmological redshift |
| `t0` | free | Time of B-band maximum (MJD) |
| `x0` | free | Overall amplitude — related to luminosity distance |
| `x1` | free | Stretch — encodes light-curve width / decline rate |
| `c`  | free | Colour — encodes SED tilt (dust + intrinsic colour) |

### SALT3 vs SALT2 in practice

- **Same number of free parameters**: no extra parameter in SALT3.
- **Better error estimates**: SALT3 uncertainties (especially σ(x1) and σ(c))
  are more reliable because they are computed in flux space.
- **Native y-band coverage**: SALT3 is trained up to 11 000 Å, so the LSST y-band
  (λ_eff ≈ 9 700 Å) is within the trained model range — no extrapolation needed.
- **Larger, better-calibrated training set**: 1 083 SNe vs ~430 for SALT2.4.

### Further extension — SALT3-NIR

For NIR observations (e.g. Roman Space Telescope) beyond LSST's coverage,
the `'salt3-nir'` model extends the template to 2 μm.  
It is **not required** for LSST since all six LSST bands lie within SALT3.K21's
2 000–11 000 Å range.
